# Chroma 고급 검색 — metadata filter / hybrid / re-ranking — 복원본 (대화 기반 재구성)

> ⚠️ 원본이 저장 누락으로 0바이트가 돼서, Claude와의 학습 대화에 남은 코드·구조로 **재구성**한 거야.
> 네가 그때 짠 원본 그대로는 아니니 **한 번 검토하고 직접 재실행**해. `TODO/확인` 자리는 네 데이터/환경에 맞춰 채우면 돼.
> LLM 생성이 필요한 셀은 무료 Groq API 키가 필요해 (OpenAI 크레딧 소진 상태).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma

docs = [
    "LLM은 이전 토큰들을 보고 다음 토큰의 확률을 예측하는 모델이다.",
    "임베딩은 텍스트의 의미를 벡터로 바꾸며, 의미가 비슷할수록 벡터가 가깝다.",
    "RAG는 관련 문서를 검색해 프롬프트에 넣고, 그것을 근거로 답을 생성한다.",
    "벡터 DB는 수많은 임베딩 중 질문과 가장 가까운 것을 빠르게 찾는다.",
    "파인튜닝은 모델의 가중치를 직접 학습시켜 행동이나 도메인을 바꾼다.",
    "프롬프트 엔지니어링은 가중치를 바꾸지 않고, 입력 맥락으로 출력을 유도한다.",
]
topics = ["LLM", "임베딩", "RAG", "벡터DB", "파인튜닝", "프롬프트"]

documents = [Document(page_content=t, metadata={"topic": tp})
             for t, tp in zip(docs, topics)]
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = Chroma.from_documents(documents, embedding=embeddings)

### 1) Metadata 필터
`filter`는 **유사도 계산 전에** metadata로 후보를 걸러냄 → "이 조건 맞는 문서 안에서만 가까운 거". 검색 *범위*를 좁히는 것.

In [ ]:
# 필터 없이 vs 있이 비교
print("[필터 없음]")
for r in vectorstore.similarity_search("모델 가중치를 학습", k=3):
    print(" ", r.metadata["topic"], "|", r.page_content)

print("[필터: topic=프롬프트]")
for r in vectorstore.similarity_search("모델 가중치를 학습", k=3, filter={"topic": "프롬프트"}):
    print(" ", r.metadata["topic"], "|", r.page_content)
# 포인트: filter가 후보를 먼저 자르므로, 해당 topic 문서가 1개면 k=3이어도 1개만 나옴

### 2) Hybrid (BM25 + dense)
dense(임베딩)는 *의미*에 강하지만 정확한 키워드(고유명사·약어)를 놓침. BM25(키워드)는 그 반대.
→ 둘을 **RRF**(순위 기반 결합)로 합쳐 약점 보완. `pip install rank_bm25` 필요.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

bm25 = BM25Retriever.from_documents(documents)   # sparse(키워드)
bm25.k = 3
dense = vectorstore.as_retriever(search_kwargs={"k": 3})

ensemble = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])
for r in ensemble.invoke("RAG"):
    print(r.metadata["topic"], "|", r.page_content)
# 한국어 주의: BM25 기본 토크나이저는 공백 기준이라 조사가 붙음("RAG는"≠"RAG").
#   제대로 하려면 preprocess_func에 형태소 분석기(Kiwi/Okt). 데모는 그냥 돌아감.

### 3) Re-ranking (cross-encoder) — ★ 오늘(Week 10) 일정과 직결
1차 검색으로 후보 N개 넓게 뽑고, cross-encoder가 (질문, 문서) 쌍을 직접 보고 정밀 재정렬.
bi-encoder(임베딩)는 따로 인코딩해 빠르지만 거칠고, cross-encoder는 같이 보고 정확하지만 느림 → 후보에만 적용.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
query = "검색 결과를 근거로 답을 만드는 방식"

# 1차: dense로 후보 넓게
cands = [d.page_content for d in vectorstore.similarity_search(query, k=5)]

# TODO: (query, 후보) 쌍 점수 매기고 내림차순 정렬
#   힌트: scores = reranker.predict([(query, c) for c in cands])
#         ranked = [c for _, c in sorted(zip(scores, cands), reverse=True)]
ranked = None   # <-- 채우기
print(ranked)

### 정리
- **metadata filter**: 검색 범위를 조건으로 좁힘
- **hybrid**: 의미(dense)+키워드(BM25)를 RRF로 결합
- **re-rank**: 넓게 뽑고 cross-encoder로 정밀 재정렬 → 정밀도↑
오늘 rag_app에 붙일 rerank가 바로 3번이야.